# Taller — Pruebas de Hipótesis en R
## Caso de Estudio: Control de Calidad en NutriFood S.A.

| | |
|---|---|
| **Curso:** | Estadística II |
| **Tema:** | Contrastes de hipótesis — una y dos poblaciones |
| **Herramienta:** | R |
| **Nombre:** | ________________________________ |
| **Fecha:** | ________________________________ |

---

## Contexto del caso

**NutriFood S.A.** es una empresa productora de barras nutricionales que debe cumplir estrictos estándares de calidad ante la autoridad sanitaria. Cada barra está declarada con un peso neto de **50 g**. El departamento de control de calidad dispone de los siguientes datos:

- **Línea A** (`peso_A`): muestra de 25 barras tomada durante el turno de la mañana.  
- **Línea B** (`peso_B`): muestra de 20 barras de una segunda línea de producción.  
- **Línea C** (`peso_C`): muestra de 15 barras de una tercera línea con maquinaria más antigua.  
- **Proceso (antes/después)** (`antes`, `despues`): 20 mediciones pareadas del mismo lote, tomadas antes y después de una calibración del equipo.  
- **Defectos**: en una inspección de 200 unidades se encontraron 8 barras fuera de especificación.

A lo largo del taller se construirán y resolverán contrastes de hipótesis para cada escenario, implementando manualmente los estadísticos de prueba y verificando con las funciones nativas de R.

---

> **Nota metodológica:** El nivel de significancia empleado en todo el taller es $\alpha = 0.05$, salvo indicación contraria.

In [ ]:
# ── Configuración inicial ────────────────────────────────────────────────────
set.seed(42)

# Línea A: n = 25 barras (turno mañana)
peso_A <- rnorm(25, mean = 50.8, sd = 2.1)

# Línea B: n = 20 barras (segunda línea)
peso_B <- rnorm(20, mean = 49.5, sd = 1.9)

# Línea C: n = 15 barras (maquinaria antigua, mayor variabilidad)
peso_C <- rnorm(15, mean = 52.3, sd = 3.5)

# Mediciones ANTES y DESPUÉS de la calibración (20 pares del mismo lote)
antes   <- rnorm(20, mean = 49.5, sd = 2.0)
despues <- antes + rnorm(20, mean = 0.9, sd = 1.1)

# Defectos en muestra de tamaño grande
n_inspeccion <- 200
n_defectos   <- 8

# Parámetro histórico de referencia
mu_0    <- 50       # peso declarado (g)
sigma_0 <- 2.0      # desviación estándar histórica conocida del proceso
sigma2_0 <- sigma_0^2

# ── Resumen exploratorio ────────────────────────────────────────────────────
cat("── Estadísticos descriptivos ──────────────────────────────\n")
cat(sprintf("Línea A : n=%d  x̄=%.4f  S=%.4f\n", length(peso_A), mean(peso_A), sd(peso_A)))
cat(sprintf("Línea B : n=%d  x̄=%.4f  S=%.4f\n", length(peso_B), mean(peso_B), sd(peso_B)))
cat(sprintf("Línea C : n=%d  x̄=%.4f  S=%.4f\n", length(peso_C), mean(peso_C), sd(peso_C)))
cat(sprintf("Antes   : n=%d  x̄=%.4f  S=%.4f\n", length(antes),  mean(antes),  sd(antes)))
cat(sprintf("Después : n=%d  x̄=%.4f  S=%.4f\n", length(despues),mean(despues),sd(despues)))
cat(sprintf("Defectos: %d / %d  p̂=%.4f\n", n_defectos, n_inspeccion,
            n_defectos / n_inspeccion))

---
## 1. Contraste para la Media — Varianza Poblacional Conocida (Prueba Z)

**Escenario:** El proceso de llenado de la Línea A tiene una desviación estándar histórica conocida $\sigma = 2$ g, establecida por el fabricante del equipo. El jefe de producción sospecha que el peso promedio se ha desviado del estándar de 50 g.

$$H_0: \mu = 50 \quad \text{vs} \quad H_1: \mu \neq 50$$

**Estadístico de prueba:**

$$Z = \frac{\bar{X} - \mu_0}{\sigma / \sqrt{n}} \sim N(0,1) \quad \text{bajo } H_0$$

**Regla de decisión:** Rechazar $H_0$ si $|Z| > z_{\alpha/2} = z_{0.025} = 1.960$.

In [ ]:
# ── Contraste Z para la media (σ conocida) ───────────────────────────────────
n1    <- length(peso_A)
xbar1 <- mean(peso_A)
alpha <- 0.05

# Estadístico de prueba
Z_calc <- (xbar1 - mu_0) / (sigma_0 / sqrt(n1))

# Valor crítico y valor p (bilateral)
z_crit <- qnorm(1 - alpha / 2)
p_val  <- 2 * pnorm(-abs(Z_calc))

cat("── Prueba Z bilateral ──────────────────────────────────────\n")
cat(sprintf("  n = %d  |  x̄ = %.4f  |  σ = %.1f\n", n1, xbar1, sigma_0))
cat(sprintf("  Z calculado  = %.4f\n", Z_calc))
cat(sprintf("  Valor crítico = ±%.4f\n", z_crit))
cat(sprintf("  Valor p       = %.4f\n", p_val))
cat(sprintf("  Decisión      : %s\n",
            ifelse(abs(Z_calc) > z_crit, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Visualización de la región crítica ──────────────────────────────────────
x_seq <- seq(-4, 4, length.out = 400)
y_seq <- dnorm(x_seq)

plot(x_seq, y_seq, type = "l", lwd = 2, col = "navy",
     xlab = "Z", ylab = "Densidad",
     main = expression(paste("Prueba Z bilateral  —  H[0]: mu==50")))

# Regiones críticas (sombreado)
x_izq <- x_seq[x_seq <= -z_crit]
x_der <- x_seq[x_seq >=  z_crit]
polygon(c(-4, x_izq, -z_crit), c(0, dnorm(x_izq), 0), col = "#FF6B6B88", border = NA)
polygon(c( z_crit, x_der,  4), c(0, dnorm(x_der), 0), col = "#FF6B6B88", border = NA)

# Estadístico observado
abline(v = Z_calc, col = "darkgreen", lwd = 2, lty = 2)
abline(v = c(-z_crit, z_crit), col = "red", lwd = 1.5, lty = 3)

legend("topright", bty = "n",
       legend = c(sprintf("Z = %.3f", Z_calc), sprintf("±z₀.₀₂₅ = ±%.3f", z_crit), "Región crítica"),
       col    = c("darkgreen", "red", "#FF6B6B"),
       lty    = c(2, 3, NA), pch = c(NA, NA, 15), lwd = 2)

---
## 2. Contraste para la Media — Varianza Poblacional Desconocida (Prueba t)

**Escenario:** En un nuevo turno, la desviación estándar del proceso no puede garantizarse; sólo se dispone de la varianza muestral $S$ de la Línea A.

$$H_0: \mu = 50 \quad \text{vs} \quad H_1: \mu > 50$$

**Estadístico de prueba:**

$$t = \frac{\bar{X} - \mu_0}{S / \sqrt{n}} \sim t_{n-1} \quad \text{bajo } H_0$$

**Regla de decisión (unilateral derecha):** Rechazar $H_0$ si $t > t_{\alpha,\, n-1}$.

In [ ]:
# ── Contraste t para la media (σ desconocida) ────────────────────────────────
S1     <- sd(peso_A)
t_calc <- (xbar1 - mu_0) / (S1 / sqrt(n1))
t_crit <- qt(1 - alpha, df = n1 - 1)   # unilateral derecha
p_t    <- pt(t_calc, df = n1 - 1, lower.tail = FALSE)

cat("── Prueba t unilateral (derecha) ───────────────────────────\n")
cat(sprintf("  n = %d  |  x̄ = %.4f  |  S = %.4f\n", n1, xbar1, S1))
cat(sprintf("  t calculado   = %.4f\n", t_calc))
cat(sprintf("  t crítico     = %.4f  (t_0.05, gl=%d)\n", t_crit, n1 - 1))
cat(sprintf("  Valor p       = %.4f\n", p_t))
cat(sprintf("  Decisión      : %s\n",
            ifelse(t_calc > t_crit, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Verificación con t.test() ────────────────────────────────────────────────
cat("\n── Verificación con t.test() ───────────────────────────────\n")
print(t.test(peso_A, mu = mu_0, alternative = "greater"))

---
## 3. Contraste para una Proporción Poblacional

**Escenario:** El estándar regulatorio exige que la tasa de unidades defectuosas no supere el 5 %. En la inspección de 200 barras se encontraron 8 defectuosas ($\hat{p} = 0.04$). El equipo de calidad quiere demostrar que la tasa real es inferior al umbral.

$$H_0: P = 0.05 \quad \text{vs} \quad H_1: P < 0.05$$

**Estadístico de prueba (muestra grande):**

$$Z = \frac{\hat{p} - P_0}{\sqrt{P_0(1-P_0)/n}} \sim N(0,1) \quad \text{bajo } H_0$$

**Regla de decisión (unilateral izquierda):** Rechazar $H_0$ si $Z < -z_{\alpha} = -1.645$.

In [ ]:
# ── Contraste Z para la proporción ──────────────────────────────────────────
P0    <- 0.05
p_hat <- n_defectos / n_inspeccion
n_p   <- n_inspeccion

Z_p    <- (p_hat - P0) / sqrt(P0 * (1 - P0) / n_p)
z_crit_izq <- -qnorm(1 - alpha)    # unilateral izquierda
p_val_p    <- pnorm(Z_p)            # P(Z ≤ Z_calc)

cat("── Prueba Z para proporción (unilateral izquierda) ─────────\n")
cat(sprintf("  n = %d  |  defectos = %d  |  p̂ = %.4f  |  P₀ = %.2f\n",
            n_p, n_defectos, p_hat, P0))
cat(sprintf("  Z calculado  = %.4f\n", Z_p))
cat(sprintf("  Z crítico    = %.4f  (-z_0.05)\n", z_crit_izq))
cat(sprintf("  Valor p      = %.4f\n", p_val_p))
cat(sprintf("  Decisión     : %s\n",
            ifelse(Z_p < z_crit_izq, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Verificación con prop.test() (con corrección de continuidad desactivada) ──
cat("\n── Verificación con prop.test() ────────────────────────────\n")
print(prop.test(n_defectos, n_inspeccion, p = P0,
                alternative = "less", correct = FALSE))

---
## 4. Contraste para la Varianza Poblacional (Distribución $\chi^2$)

**Escenario:** El proceso de llenado de la Línea A tiene una especificación de varianza máxima $\sigma^2_0 = 4$ g² (equivalente a $\sigma_0 = 2$ g). Varianzas mayores implican riesgo de incumplimiento regulatorio.

$$H_0: \sigma^2 = 4 \quad \text{vs} \quad H_1: \sigma^2 > 4$$

**Estadístico de prueba:**

$$\chi^2 = \frac{(n-1)S^2}{\sigma_0^2} \sim \chi^2_{n-1} \quad \text{bajo } H_0$$

**Regla de decisión (unilateral derecha):** Rechazar $H_0$ si $\chi^2 > \chi^2_{\alpha,\, n-1}$.

In [ ]:
# ── Contraste Chi-cuadrado para la varianza ──────────────────────────────────
S2_A  <- var(peso_A)
gl    <- n1 - 1

chi2_calc <- gl * S2_A / sigma2_0
chi2_crit <- qchisq(1 - alpha, df = gl)    # percentil 95
p_chi2    <- pchisq(chi2_calc, df = gl, lower.tail = FALSE)

cat("── Prueba χ² para la varianza (unilateral derecha) ─────────\n")
cat(sprintf("  n = %d  |  S² = %.4f  |  σ₀² = %.1f\n", n1, S2_A, sigma2_0))
cat(sprintf("  χ² calculado = %.4f\n", chi2_calc))
cat(sprintf("  χ² crítico   = %.4f  (χ²_0.05, gl=%d)\n", chi2_crit, gl))
cat(sprintf("  Valor p      = %.4f\n", p_chi2))
cat(sprintf("  Decisión     : %s\n",
            ifelse(chi2_calc > chi2_crit, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Gráfico de la distribución χ² con región crítica ────────────────────────
x_chi <- seq(0, 50, length.out = 500)
y_chi <- dchisq(x_chi, df = gl)

plot(x_chi, y_chi, type = "l", lwd = 2, col = "navy",
     xlab = expression(chi^2), ylab = "Densidad",
     main = bquote(paste("Contraste " ~ chi^2 ~ " — H"[0]*": " ~ sigma^2 == 4)))

x_rc <- x_chi[x_chi >= chi2_crit]
polygon(c(chi2_crit, x_rc, max(x_chi)),
        c(0, dchisq(x_rc, df = gl), 0),
        col = "#FF6B6B88", border = NA)

abline(v = chi2_calc, col = "darkgreen", lwd = 2, lty = 2)
abline(v = chi2_crit,  col = "red",       lwd = 1.5, lty = 3)
legend("topright", bty = "n",
       legend = c(sprintf("χ² = %.3f", chi2_calc),
                  sprintf("χ²_crit = %.3f", chi2_crit), "Región crítica"),
       col = c("darkgreen", "red", "#FF6B6B"),
       lty = c(2, 3, NA), pch = c(NA, NA, 15), lwd = 2)

---
## 5. Análisis de Potencia y Errores Tipo I y II

**Escenario:** Usando el contraste bilateral de la Sección 1 ($\sigma = 2$, $n = 25$, $\alpha = 0.05$), se desea estudiar cómo varía la potencia del test en función del valor verdadero de $\mu$.

La potencia para un valor alternativo $\mu_1$ es:

$$1 - \beta(\mu_1) = \Phi\!\left(-z_{\alpha/2} + \frac{|\mu_1 - \mu_0|}{\sigma/\sqrt{n}}\right) + \Phi\!\left(-z_{\alpha/2} - \frac{|\mu_1 - \mu_0|}{\sigma/\sqrt{n}}\right)$$

Se comparan además el efecto de distintos tamaños muestrales sobre la curva de potencia.

In [ ]:
# ── Función de potencia para prueba Z bilateral ──────────────────────────────
potencia_Z <- function(mu1, mu0 = 50, sigma = 2, n, alpha = 0.05) {
  z_a  <- qnorm(1 - alpha / 2)
  delta <- (mu1 - mu0) / (sigma / sqrt(n))
  pnorm(-z_a + delta) + pnorm(-z_a - delta)
}

mu_vals <- seq(48, 52, length.out = 300)
ns      <- c(10, 25, 50, 100)
colores <- c("#E74C3C", "#2ECC71", "#3498DB", "#9B59B6")

# ── Curvas de potencia ───────────────────────────────────────────────────────
plot(mu_vals, potencia_Z(mu_vals, n = ns[1]),
     type = "l", lwd = 2, col = colores[1], ylim = c(0, 1),
     xlab = expression(mu ~ "(valor verdadero)"),
     ylab = expression(paste("Potencia  ", 1 - beta)),
     main = "Curvas de potencia — Prueba Z bilateral (σ=2, α=0.05)")

for (i in 2:length(ns)) {
  lines(mu_vals, potencia_Z(mu_vals, n = ns[i]),
        lwd = 2, col = colores[i])
}

abline(v  = mu_0,  col = "gray40", lty = 2)
abline(h  = alpha, col = "gray40", lty = 3)
abline(h  = 0.80,  col = "black",  lty = 3, lwd = 1.2)

legend("bottomright", bty = "n",
       legend = paste("n =", ns),
       col = colores, lwd = 2)
text(52.05, 0.82, "Potencia = 0.80", cex = 0.8)
text(50.05, 0.06, "H₀: μ=50", cex = 0.8, col = "gray40")

# ── Error Tipo II para n=25 y μ₁=51 ────────────────────────────────────────
mu1_ref  <- 51
pot_ref  <- potencia_Z(mu1_ref, n = 25)
beta_ref <- 1 - pot_ref

cat("── Resumen para μ₁ = 51, n = 25, α = 0.05 ─────────────────\n")
cat(sprintf("  Potencia (1-β) = %.4f\n", pot_ref))
cat(sprintf("  Error Tipo II (β) = %.4f\n", beta_ref))

# ── Tamaño muestral necesario para potencia ≥ 0.90 (μ₁=51) ─────────────────
n_seq  <- 5:200
pot_n  <- sapply(n_seq, function(n) potencia_Z(mu1_ref, n = n))
n_90   <- n_seq[which(pot_n >= 0.90)[1]]
cat(sprintf("  n mínimo para potencia ≥ 0.90 detectando μ=51: n = %d\n", n_90))

---
## 6. Contraste para Dos Medias — Muestras Dependientes (Enlazadas)

**Escenario:** Se toman 20 barras del mismo lote y se pesan **antes** y **después** de calibrar la báscula de llenado. Al ser las mismas unidades, las muestras están emparejadas. Se quiere verificar si la calibración produjo un cambio significativo en el peso registrado.

$$H_0: \mu_D = 0 \quad \text{vs} \quad H_1: \mu_D \neq 0, \qquad D_i = \text{Después}_i - \text{Antes}_i$$

**Estadístico de prueba:**

$$t = \frac{\bar{D}}{S_D / \sqrt{n}} \sim t_{n-1} \quad \text{bajo } H_0$$

In [ ]:
# ── Contraste t para muestras dependientes ──────────────────────────────────
D    <- despues - antes
Dbar <- mean(D)
SD   <- sd(D)
nD   <- length(D)

t_D     <- Dbar / (SD / sqrt(nD))
t_crit2 <- qt(1 - alpha / 2, df = nD - 1)   # bilateral
p_D     <- 2 * pt(-abs(t_D), df = nD - 1)

cat("── Prueba t pareada bilateral ───────────────────────────────\n")
cat(sprintf("  n pares = %d  |  D̄ = %.4f  |  S_D = %.4f\n", nD, Dbar, SD))
cat(sprintf("  t calculado  = %.4f\n", t_D))
cat(sprintf("  t crítico    = ±%.4f  (t_0.025, gl=%d)\n", t_crit2, nD - 1))
cat(sprintf("  Valor p      = %.4f\n", p_D))
cat(sprintf("  Decisión     : %s\n",
            ifelse(abs(t_D) > t_crit2, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Visualización de las diferencias ────────────────────────────────────────
par(mfrow = c(1, 2))

boxplot(D, col = "#85C1E9", main = "Distribución de diferencias D",
        ylab = "Después − Antes (g)")
abline(h = 0, col = "red", lty = 2, lwd = 2)

plot(antes, despues, pch = 19, col = "#2ECC71",
     xlab = "Peso antes (g)", ylab = "Peso después (g)",
     main = "Antes vs Después (pares)")
abline(0, 1, col = "red", lty = 2, lwd = 2)   # línea de igualdad

par(mfrow = c(1, 1))

# ── Verificación con t.test(paired=TRUE) ────────────────────────────────────
cat("\n── Verificación con t.test(paired=TRUE) ────────────────────\n")
print(t.test(despues, antes, paired = TRUE, alternative = "two.sided"))

---
## 7. Igualdad de Varianzas entre dos Poblaciones (Distribución F)

**Escenario:** Antes de comparar las medias de la Línea A y la Línea B mediante una prueba $t$, es necesario verificar si sus varianzas son estadísticamente iguales, lo cual determina qué versión del test usar.

$$H_0: \sigma_A^2 = \sigma_B^2 \quad \text{vs} \quad H_1: \sigma_A^2 \neq \sigma_B^2$$

**Estadístico de prueba:**

$$F = \frac{S_A^2}{S_B^2} \sim F_{n_A-1,\; n_B-1} \quad \text{bajo } H_0$$

**Regla de decisión (bilateral):** Rechazar $H_0$ si $F < F_{\alpha/2,\, gl_1,\, gl_2}$ o $F > F_{1-\alpha/2,\, gl_1,\, gl_2}$.

In [ ]:
# ── Prueba F de igualdad de varianzas (Línea A vs Línea B) ──────────────────
n2   <- length(peso_B)
S2_B <- var(peso_B)
gl1  <- n1 - 1
gl2  <- n2 - 1

F_calc  <- S2_A / S2_B
F_lo    <- qf(alpha / 2, df1 = gl1, df2 = gl2)      # percentil 2.5%
F_hi    <- qf(1 - alpha / 2, df1 = gl1, df2 = gl2)  # percentil 97.5%
p_F     <- 2 * min(pf(F_calc, df1 = gl1, df2 = gl2),
                   pf(F_calc, df1 = gl1, df2 = gl2, lower.tail = FALSE))

cat("── Prueba F bilateral (Línea A vs Línea B) ─────────────────\n")
cat(sprintf("  S²_A = %.4f  (n=%d)  |  S²_B = %.4f  (n=%d)\n",
            S2_A, n1, S2_B, n2))
cat(sprintf("  F calculado  = %.4f\n", F_calc))
cat(sprintf("  F crítico    [%.4f , %.4f]  (gl=%d,%d)\n",
            F_lo, F_hi, gl1, gl2))
cat(sprintf("  Valor p      = %.4f\n", p_F))
cat(sprintf("  Decisión     : %s → ",
            ifelse(F_calc < F_lo | F_calc > F_hi, "RECHAZAR H₀", "NO rechazar H₀")))
cat(ifelse(F_calc < F_lo | F_calc > F_hi,
           "varianzas DESIGUALES: usar Welch.\n",
           "varianzas IGUALES: se puede usar t pooled.\n"))

# ── Verificación con var.test() ──────────────────────────────────────────────
cat("\n── Verificación con var.test() ─────────────────────────────\n")
print(var.test(peso_A, peso_B, alternative = "two.sided"))

---
## 8. Contraste de Medias — Muestras Independientes, Varianzas Iguales (t Pooled)

**Escenario:** Con base en el resultado de la Sección 7 (varianzas estadísticamente iguales entre Líneas A y B), se contrasta si las medias de producción de ambas líneas son equivalentes.

$$H_0: \mu_A = \mu_B \quad \text{vs} \quad H_1: \mu_A \neq \mu_B$$

**Estimador combinado de la varianza (pooled):**

$$S_p^2 = \frac{(n_A - 1)S_A^2 + (n_B - 1)S_B^2}{n_A + n_B - 2}$$

**Estadístico de prueba:**

$$t = \frac{\bar{X}_A - \bar{X}_B}{S_p\sqrt{1/n_A + 1/n_B}} \sim t_{n_A+n_B-2} \quad \text{bajo } H_0$$

In [ ]:
# ── t-test con varianzas iguales (pooled) — Línea A vs Línea B ───────────────
xbar_A <- mean(peso_A)
xbar_B <- mean(peso_B)
Sp2    <- ((n1 - 1) * S2_A + (n2 - 1) * S2_B) / (n1 + n2 - 2)
Sp     <- sqrt(Sp2)
gl_p   <- n1 + n2 - 2

t_pool  <- (xbar_A - xbar_B) / (Sp * sqrt(1/n1 + 1/n2))
t_crit_p <- qt(1 - alpha / 2, df = gl_p)
p_pool   <- 2 * pt(-abs(t_pool), df = gl_p)

cat("── Prueba t bilateral con varianzas iguales (Línea A vs B) ─\n")
cat(sprintf("  x̄_A = %.4f  |  x̄_B = %.4f  |  Sp² = %.4f  |  gl = %d\n",
            xbar_A, xbar_B, Sp2, gl_p))
cat(sprintf("  t calculado  = %.4f\n", t_pool))
cat(sprintf("  t crítico    = ±%.4f\n", t_crit_p))
cat(sprintf("  Valor p      = %.4f\n", p_pool))
cat(sprintf("  IC 95%%: (%.4f , %.4f)\n",
            (xbar_A - xbar_B) - t_crit_p * Sp * sqrt(1/n1 + 1/n2),
            (xbar_A - xbar_B) + t_crit_p * Sp * sqrt(1/n1 + 1/n2)))
cat(sprintf("  Decisión     : %s\n",
            ifelse(abs(t_pool) > t_crit_p, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Verificación con t.test(var.equal=TRUE) ──────────────────────────────────
cat("\n── Verificación con t.test(var.equal=TRUE) ─────────────────\n")
print(t.test(peso_A, peso_B, var.equal = TRUE, alternative = "two.sided"))

# ── Diagrama comparativo ─────────────────────────────────────────────────────
boxplot(list("Línea A" = peso_A, "Línea B" = peso_B),
        col = c("#85C1E9", "#A9DFBF"),
        ylab = "Peso (g)", main = "Comparación Línea A vs Línea B",
        outline = TRUE)
abline(h = mu_0, col = "red", lty = 2, lwd = 2)
legend("topright", bty = "n", legend = "μ₀ = 50 g",
       col = "red", lty = 2, lwd = 2)

---
## 9. Contraste de Medias — Muestras Independientes, Varianzas Desiguales (Welch)

**Escenario:** La Línea C usa maquinaria más antigua con mayor variabilidad ($S_C \approx 3.5$ g). Al comparar Línea A vs Línea C, la prueba F preliminar revela varianzas significativamente distintas, por lo que se aplica la aproximación de **Welch-Satterthwaite**.

$$H_0: \mu_A = \mu_C \quad \text{vs} \quad H_1: \mu_A \neq \mu_C$$

**Estadístico y grados de libertad aproximados:**

$$t_W = \frac{\bar{X}_A - \bar{X}_C}{\sqrt{S_A^2/n_A + S_C^2/n_C}}, \qquad \nu \approx \frac{\left(\dfrac{S_A^2}{n_A} + \dfrac{S_C^2}{n_C}\right)^2}{\dfrac{(S_A^2/n_A)^2}{n_A-1} + \dfrac{(S_C^2/n_C)^2}{n_C-1}}$$

In [ ]:
# ── Prueba F previa: Línea A vs Línea C ─────────────────────────────────────
n3   <- length(peso_C)
S2_C <- var(peso_C)

cat("── Prueba F previa (Línea A vs Línea C) ────────────────────\n")
vt <- var.test(peso_A, peso_C, alternative = "two.sided")
cat(sprintf("  S²_A = %.4f  |  S²_C = %.4f\n", S2_A, S2_C))
cat(sprintf("  F = %.4f  |  valor p = %.4f → %s\n\n",
            vt$statistic, vt$p.value,
            ifelse(vt$p.value < alpha,
                   "varianzas DESIGUALES → usar Welch",
                   "varianzas IGUALES → pooled")))

# ── Welch t-test manual ───────────────────────────────────────────────────────
xbar_C <- mean(peso_C)
SE_W   <- sqrt(S2_A / n1 + S2_C / n3)

# Grados de libertad de Welch-Satterthwaite
num_gl <- (S2_A / n1 + S2_C / n3)^2
den_gl <- (S2_A / n1)^2 / (n1 - 1) + (S2_C / n3)^2 / (n3 - 1)
nu_W   <- num_gl / den_gl

t_W     <- (xbar_A - xbar_C) / SE_W
t_crit_W <- qt(1 - alpha / 2, df = nu_W)
p_W      <- 2 * pt(-abs(t_W), df = nu_W)

cat("── Prueba de Welch bilateral (Línea A vs Línea C) ──────────\n")
cat(sprintf("  x̄_A = %.4f  |  x̄_C = %.4f\n", xbar_A, xbar_C))
cat(sprintf("  SE_Welch = %.4f  |  ν_Welch = %.2f\n", SE_W, nu_W))
cat(sprintf("  t_W calculado = %.4f\n", t_W))
cat(sprintf("  t crítico     = ±%.4f\n", t_crit_W))
cat(sprintf("  Valor p       = %.4f\n", p_W))
cat(sprintf("  Decisión      : %s\n",
            ifelse(abs(t_W) > t_crit_W, "RECHAZAR H₀", "NO rechazar H₀")))

# ── Verificación con t.test() (Welch es el default) ──────────────────────────
cat("\n── Verificación con t.test() (Welch, var.equal=FALSE) ──────\n")
print(t.test(peso_A, peso_C, var.equal = FALSE, alternative = "two.sided"))

---
## Resumen de Conclusiones

| # | Contraste | $H_0$ | $H_1$ | Distribución | Decisión |
|:---:|---|---|---|:---:|:---:|
| 1 | Media $\mu$ — $\sigma$ conocida (Z) | $\mu = 50$ | $\mu \neq 50$ | $N(0,1)$ | *(ejecutar)* |
| 2 | Media $\mu$ — $\sigma$ desconocida (t) | $\mu = 50$ | $\mu > 50$ | $t_{24}$ | *(ejecutar)* |
| 3 | Proporción $P$ | $P = 0.05$ | $P < 0.05$ | $N(0,1)$ | *(ejecutar)* |
| 4 | Varianza $\sigma^2$ | $\sigma^2 = 4$ | $\sigma^2 > 4$ | $\chi^2_{24}$ | *(ejecutar)* |
| 5 | Potencia (curvas) | — | — | $N(0,1)$ | *(gráfico)* |
| 6 | Medias dependientes (pareado) | $\mu_D = 0$ | $\mu_D \neq 0$ | $t_{19}$ | *(ejecutar)* |
| 7 | Igualdad varianzas A vs B (F) | $\sigma_A^2 = \sigma_B^2$ | $\neq$ | $F_{24,19}$ | *(ejecutar)* |
| 8 | Medias indep. var. iguales (pooled) | $\mu_A = \mu_B$ | $\neq$ | $t_{43}$ | *(ejecutar)* |
| 9 | Medias indep. var. desiguales (Welch) | $\mu_A = \mu_C$ | $\neq$ | $t_\nu$ (Welch) | *(ejecutar)* |

---

> **Nota:** La columna "Decisión" se completa automáticamente al ejecutar las celdas de código. Los resultados dependen de la semilla `set.seed(42)` fijada al inicio.